<a href="https://colab.research.google.com/github/iHakawaTi/Crown-Prince-App/blob/main/Synthetic_Data_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.4 MB/s eta 0:00:00


In [28]:
from faker import Faker
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import hashlib

# Important notes:-
# This code is specific to Jordanian standard blood donors
# Constraints on data have been put according to google search and other relevant statistics
# This code uses fake data, nothing here can be linked to PII
# Using the faker library with constraints to generate somewhat realistic data


class MedicalDonorDataGenerator:
    def __init__(self, n_samples=1000, seed=42):
        self.n_samples = n_samples
        self.fake = Faker('ar')  # Arabic
        np.random.seed(seed)
        Faker.seed(seed)

        # Define Medical constraints as config
        # i used google for the constraints (they prolly wont be as accurate)
        self.constraints = {

            # Age: Standard eligibility 18-65; mean ~35 for active donor pool (most donors 20-45)
            'age': {'min': 18, 'max': 65, 'mean': 35, 'std': 10},

            # Weight: Min 50kg for whole blood; mean ~75kg typical adult in Middle East/Jordan
            'weight': {'min': 50, 'max': 120, 'mean': 75, 'std': 15},

            # Hemoglobin Males: Min 12.5 g/dL eligibility; mean 13.5-14.5 for active donors
            'hemoglobin_male': {'min': 12.5, 'max': 17.5, 'mean': 14, 'std': 1},

            # Hemoglobin Females: Min 12.5 g/dL eligibility (some sources say 12.0); mean 12.5-13.5
            'hemoglobin_female': {'min': 12.0, 'max': 16.0, 'mean': 13.2, 'std': 1},

            # Systolic BP: Typical limit for deferral >160 mmHg; Red Cross defers only >180
            # Normal donor mean ~120; std ~12-15
            'systolic_bp': {'min': 90, 'max': 160, 'mean': 120, 'std': 12},

            # Diastolic BP: Typical limit <100 mmHg; Red Cross defers only >100
            # Normal mean ~80; std ~8-10
            'diastolic_bp': {'min': 60, 'max': 100, 'mean': 80, 'std': 8},

            # Pulse: Standard 60-100 bpm eligible; mean ~72-75 normal
            'pulse': {'min': 60, 'max': 100, 'mean': 72, 'std': 8},
        }

        # Blood type distribution for Jordan
        # Source: Wikipedia - Blood Type Distribution by Country, Jordan Journal of Medical Sciences
        # Jordan specific: O+ 33.03%, A+ 32.86%, B+ 16.56%, AB+ 6.28%, O- 4.4%, A- 3.97%, B- 2.06%, AB- 0.04%
        self.blood_groups = ['O+', 'A+', 'B+', 'AB+', 'O-', 'A-', 'B-', 'AB-']
        self.blood_group_probabilities = [0.3338, 0.3307, 0.1668, 0.0633, 0.0443, 0.0400, 0.0207, 0.0004]


        # Major Jordan cities by population (2026 estimates)
        # Source: World Population Review, Citypopulation.de
        # Amman: 2,295,250 (44.8%), Zarqa: 765,303 (14.9%), Irbid: 571,259 (11.1%),
        # Aqaba: 95,048 (1.9%), Madaba: 82,335 (1.6%), Jerash: 27,046 (0.5%), others ~25%
        self.jordan_cities = ['Amman', 'Zarqa', 'Irbid', 'Aqaba', 'Madaba', 'Jerash', 'Other']
        self.jordan_city_probabilities = [0.448, 0.149, 0.111, 0.019, 0.016, 0.005, 0.252]  # Last is "other cities" added to fix possible skewness of data
        # City GPS - VERIFIED from latlong.info, Wikipedia, latitude.to
        # Source: latlong.info, latitude.to, Wikipedia Geography of Jordan
        self.city_centers = {
            'Amman': (31.95522, 35.94503),        # Official: 31.9552°N, 35.945°E
            'Zarqa': (32.0809, 36.1059),          # 32.080953°N, 36.1058544°E
            'Irbid': (32.5556, 35.8500),          # 32.5556°N, 35.8500°E
            'Aqaba': (29.5321, 35.0063),          # 29.5321°N, 35.0063°E
            'Madaba': (31.7276, 35.8012),         # 31.7276°N, 35.8012°E
            'Jerash': (32.2747, 35.8961),         # 32.2747°N, 35.8961°E
            'Other': (31.0000, 36.0000),          # Jordan geographic center: 31°N, 36°E , THIS IS MADE FOR OTHERS CITIES, IT ISNT REALISTIC BUT OH WELL
            # TODO: add other long/lat for others if needed
        }

    def _generate_constrained_normal(self, constraint_key, size=None):
        """Generate values from truncated normal distribution"""
        c = self.constraints[constraint_key]
        values = np.random.normal(c['mean'], c['std'], size or self.n_samples)
        return np.clip(values, c['min'], c['max'])

    def _generate_bmi(self, weight, height):
        """Calculate BMI from weight and height"""
        return weight / ((height / 100) ** 2)

    def _generate_jordan_coordinates(self, city):
        """Generate realistic coordinates within a specific Jordan city with radius variation"""
        center_lat, center_lon = self.city_centers[city]

        # Add realistic variance based on city size (in degrees, ~0.01° ≈ 1.1 km)
        city_radius = {
            'Amman': 0.15,      # ~16.5 km radius (large sprawl)
            'Zarqa': 0.08,      # ~9 km radius
            'Irbid': 0.08,      # ~9 km radius
            'Aqaba': 0.05,      # ~5.5 km radius (coastal, compact)
            'Madaba': 0.04,     # ~4.5 km radius (smaller)
            'Jerash': 0.03,     # ~3.3 km radius (small)
            'Other': 0.50       # Wide variance for other cities across Jordan
        }

        radius = city_radius.get(city, 0.05)

        # Generate random offset within circle (uniform distribution)
        angle = np.random.uniform(0, 2 * np.pi)
        r = radius * np.sqrt(np.random.uniform(0, 1))  # sqrt for uniform area distribution

        lat_offset = r * np.cos(angle)
        lon_offset = r * np.sin(angle)

        lat = round(center_lat + lat_offset, 6)
        lon = round(center_lon + lon_offset, 6)

        return lat, lon

    def _generate_donation_history(self):
        """Generate realistic donation patterns with eligibility constraints"""
        # Poisson distribution but capped - most donors have 1-5 donations
        # Mean ~2-3 for typical donor pool (mix of new and repeat donors)
        total_donations = np.random.poisson(2.5, self.n_samples)
        total_donations = np.clip(total_donations, 0, 50)

        # Days since last donation - MUST be minimum 56 days (8 weeks) for whole blood
        # Exponential with higher mean (~120 days average gap)
        days_since_last = np.random.exponential(120, self.n_samples).astype(int)
        days_since_last = np.clip(days_since_last, 56, 730)  # Min 56 days, max 2 years

        # For donors with 0 donations, set days_since_last to a high value
        days_since_last = np.where(total_donations == 0, 365, days_since_last)

        # Months since first donation - should be >= (total_donations - 1) * 2 minimum
        # Account for 8-week (2-month) minimum intervals
        min_months = (total_donations - 1) * 2
        months_since_first = np.random.randint(0, 120, self.n_samples)
        months_since_first = np.maximum(months_since_first, min_months)
        months_since_first = np.where(total_donations == 0, 0, months_since_first)

        return total_donations, days_since_last, months_since_first

    def _generate_jordan_mobile(self):
        """Generate valid Jordanian mobile number"""
        prefixes = ['077', '078', '079']
        return f'{np.random.choice(prefixes)}{np.random.randint(1000000, 9999999)}'

    def _generate_medications(self):
        """Generate realistic medication list or None"""
        if np.random.random() < 0.3:  # 30% on medications
            meds = np.random.choice(['Aspirin', 'Metformin', 'Lisinopril', 'Atorvastatin'],
                                   size=np.random.randint(1, 3), replace=False)
            return ', '.join(meds)
        return 'None'

    def _generate_response_history(self, total_donations, show_rate):
        """Generate response history array correlated with show_rate"""
        if total_donations == 0:
            return '[]'
        # Use show_rate to determine probability of responding
        # show_rate represents donor reliability, so use it directly
        prob_respond = max(0.1, min(0.95, show_rate))  # Clip between 10-95%
        responses = np.random.choice(['responded', 'no_response'],
                                    size=total_donations,
                                    p=[prob_respond, 1 - prob_respond])
        return str(responses.tolist())

    def generate(self):
        """Main generation method"""
        data = []

        for i in range(self.n_samples):
            age = int(self._generate_constrained_normal('age', 1)[0])

            # JORDAN-SPECIFIC DEMOGRAPHICS
            # Source: PMC11159535 (Jordan survey 2024) - "more than 90% of donors are male"
            if age < 30:
                gender = np.random.choice(['Male', 'Female'], p=[0.88, 0.12])  # Young peak, 88% male
            elif age < 45:
                gender = np.random.choice(['Male', 'Female'], p=[0.92, 0.08])  # 92% male
            else:
                gender = np.random.choice(['Male', 'Female'], p=[0.95, 0.05])  # 95% male

            # Physical measurements
            weight = self._generate_constrained_normal('weight', 1)[0]

            # Height varies by gender - males typically taller
            # Jordan avg: males ~174cm, females ~159cm (source: Wikipedia avg heights)
            if gender == 'Male':
                height = np.random.normal(174, 8)
                height = np.clip(height, 155, 200)
            else:
                height = np.random.normal(159, 7)
                height = np.clip(height, 145, 185)

            bmi = self._generate_bmi(weight, height)

            # Vitals
            systolic = int(self._generate_constrained_normal('systolic_bp', 1)[0])
            diastolic = int(self._generate_constrained_normal('diastolic_bp', 1)[0])

            # Ensure diastolic < systolic (physiological requirement)
            if diastolic >= systolic:
                diastolic = systolic - np.random.randint(20, 40)
                diastolic = max(diastolic, 60)

            pulse = int(self._generate_constrained_normal('pulse', 1)[0])

            # Hemoglobin based on gender
            hemo_key = 'hemoglobin_male' if gender == 'Male' else 'hemoglobin_female'
            hemoglobin = round(self._generate_constrained_normal(hemo_key, 1)[0], 1)

            # Donation history
            total_donations, days_since, months_since = self._generate_donation_history()
            total_litres = round(total_donations[i] * 0.45, 2)  # ~450ml per donation

            last_donation_date = datetime.now() - timedelta(days=int(days_since[i]))

            # City selection with population-based probabilities
            city = np.random.choice(self.jordan_cities,
                                   p=self.jordan_city_probabilities)

            # Location (Jordan-specific, city-based)
            lat, lon = self._generate_jordan_coordinates(city)

            # Generate show_rate first (needed for response_history)
            show_rate = round(np.random.beta(8, 2), 2)

            record = {
                'donor_id': f'JD{str(i+1).zfill(6)}',
                'full_name': self.fake.name(), # im too lazy for this but theres a gender mismatch that needs a couple of new functions, oh duck, another #TODO: fix gender-name mismatch
                'gender': gender,
                'age': age,
                'weight_kg': round(weight, 1),
                'height_cm': round(height, 1),
                'blood_pressure': f'{systolic}/{diastolic}',
                'pulse_rate': pulse,
                'latitude': lat,
                'longitude': lon,
                'bmi': round(bmi, 2),
                'hemoglobin_g_dl': hemoglobin,
                'medications': self._generate_medications(),
                'email': self.fake.email(),
                'password_hash': hashlib.sha256(self.fake.password().encode()).hexdigest(),
                'mobile': self._generate_jordan_mobile(),
                'city': city,
                'blood_group': np.random.choice(self.blood_groups, p=self.blood_group_probabilities),
                'availability': np.random.choice(['Yes', 'No'], p=[0.45, 0.55]),
                'months_since_first_donation': int(months_since[i]),
                'days_since_last_donation': int(days_since[i]),
                'total_donations': int(total_donations[i]),
                'total_litres_donated': total_litres,
                'last_donation_date': last_donation_date.strftime('%Y-%m-%d'),
                'show_rate': show_rate,
                'avg_response_time_hours': round(np.random.exponential(4), 1),
                'response_history': self._generate_response_history(int(total_donations[i]), show_rate),
                'record_created': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            }

            data.append(record)

        return pd.DataFrame(data)
#list of possible todos
#TODO: Validate the data
#TODO: Add noise (realistic noise)
#TODO: Split blood pressure into 2 numeric columns, do it later in the preprocessing ig
#Add 5–10% missing values randomly
#Inject slight noise into vitals
#Introduce rare anomalies (out-of-range but plausible)

#MOST IMP
#Based on the tasks, create a target variable / feature, eh?
#or
#availability
#show_rate (Donor Reliability)
#total_donations (Donor Engagement)
#will_donate_next_month (Most Practical)

# Usage
generator = MedicalDonorDataGenerator(n_samples=1000)
df = generator.generate()
df.to_csv('donor_data.csv', index=False)
print(df.head())
print(df.describe())

   donor_id          full_name gender  age  weight_kg  height_cm  \
0  JD000001    جلاء بنو العريج   Male   39       72.9      165.1   
1  JD000002    بنفسج النشاشيبي   Male   48       92.7      167.1   
2  JD000003       طبّاع الدليم   Male   18       79.3      174.7   
3  JD000004  الأستاذ عفيف راجح   Male   33       87.5      163.2   
4  JD000005       حليم الزرقان   Male   37       72.3      168.3   

  blood_pressure  pulse_rate   latitude  longitude  ...  availability  \
0         123/82          80  31.936593  35.811057  ...           Yes   
1         119/75          76  31.272952  36.019881  ...           Yes   
2         114/75          68  32.049878  35.989200  ...           Yes   
3         133/79          61  32.536775  35.791296  ...           Yes   
4         111/75          60  30.996318  36.497389  ...            No   

   months_since_first_donation days_since_last_donation total_donations  \
0                           61                       58               1   
1 